# 02_annotation_import_and_hotspot_dataset_build

Imports split-specific CVAT XML annotation files from two experts and links them to `master_metadata.csv`.

**Important design decision:** clinical labels come from folders (`healthy`/`injured`). XML annotations are used for hotspot localization and expert agreement.

This notebook explicitly flags conflicts such as healthy-folder images that contain hotspot annotations.

## 1. Imports and paths

In [1]:
from pathlib import Path
import os, json, shutil, zipfile, hashlib, re, warnings
from datetime import datetime, timezone
import pandas as pd
import numpy as np


BASE_DIR = Path("/content")
PROJECT_NAME = "project_thermography_equine"
PROJECT_ROOT = BASE_DIR / PROJECT_NAME

DATA_ROOT = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_ROOT / "raw"
SPLIT_DATA_DIR = DATA_ROOT / "dataset_split"
METADATA_DIR = DATA_ROOT / "metadata"
ANNOTATIONS_DIR = DATA_ROOT / "annotations"

OUTPUT_ROOT = PROJECT_ROOT / "outputs"
CONFIG_DIR = OUTPUT_ROOT / "config"
REPORTS_DIR = OUTPUT_ROOT / "reports"
TABLES_DIR = OUTPUT_ROOT / "tables"
FIGURES_DIR = OUTPUT_ROOT / "figures"
MODELS_DIR = OUTPUT_ROOT / "models"
MODEL_SELECTION_DIR = OUTPUT_ROOT / "model_selection"

for p in [PROJECT_ROOT, DATA_ROOT, RAW_DATA_DIR, SPLIT_DATA_DIR, METADATA_DIR, ANNOTATIONS_DIR,
          OUTPUT_ROOT, CONFIG_DIR, REPORTS_DIR, TABLES_DIR, FIGURES_DIR, MODELS_DIR, MODEL_SELECTION_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Project paths initialized")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("SPLIT_DATA_DIR:", SPLIT_DATA_DIR)
print("ANNOTATIONS_DIR:", ANNOTATIONS_DIR)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

import xml.etree.ElementTree as ET

Project paths initialized
PROJECT_ROOT: /content/project_thermography_equine
SPLIT_DATA_DIR: /content/project_thermography_equine/data/dataset_split
ANNOTATIONS_DIR: /content/project_thermography_equine/data/annotations
OUTPUT_ROOT: /content/project_thermography_equine/outputs


## 2. Load metadata and configs

In [2]:
for required_filename in ["analysis_config.json", "study_protocol.json", "annotation_config.json", "master_metadata.csv"]:
    src_path = BASE_DIR / required_filename
    dst_path = CONFIG_DIR / required_filename
    if not dst_path.exists() and src_path.exists():
        shutil.copy2(src_path, dst_path)
        print(f"Copied {required_filename} from {src_path} to {dst_path}")

for required in [CONFIG_DIR / "analysis_config.json", CONFIG_DIR / "study_protocol.json", CONFIG_DIR / "annotation_config.json", CONFIG_DIR / "master_metadata.csv"]:
    if not required.exists():
        raise FileNotFoundError(f"Missing {required}. Run notebooks 00 and 01 first.")

with open(CONFIG_DIR / "analysis_config.json", "r", encoding="utf-8") as f:
    analysis_config = json.load(f)
with open(CONFIG_DIR / "study_protocol.json", "r", encoding="utf-8") as f:
    study_protocol = json.load(f)
with open(CONFIG_DIR / "annotation_config.json", "r", encoding="utf-8") as f:
    annotation_config = json.load(f)

master_df = pd.read_csv(CONFIG_DIR / "master_metadata.csv")
print("Master metadata rows:", len(master_df))
display(master_df.groupby(["split", "label_clinical"]).size().reset_index(name="n"))

Master metadata rows: 347


,split,label_clinical,n
0,test,healthy,40
1,test,pathological,13
2,train,healthy,179
3,train,pathological,63
4,valid,healthy,38
5,valid,pathological,14


## 3. Find annotation XML files

In [3]:


def find_xml(split, expert_num):
    canonical = f"annotations_{split}_expert{expert_num}.xml"
    search_roots = [ANNOTATIONS_DIR, CONFIG_DIR, BASE_DIR, PROJECT_ROOT, DATA_ROOT]
    for root in search_roots:
        p = root / canonical
        if p.exists():
            return p
    patterns = [f"annotations_{split}_expert{expert_num}*.xml", f"*{split}*expert{expert_num}*.xml"]
    candidates = []
    for root in search_roots:
        for pat in patterns:
            candidates.extend(root.glob(pat))
    candidates = sorted(set(candidates), key=lambda p: (p.stat().st_mtime, str(p)), reverse=True)
    if candidates:
        return candidates[0]
    return None

xml_paths = []
for split in analysis_config["splits"]:
    for expert_num in [1, 2]:
        p = find_xml(split, expert_num)
        xml_paths.append({"split": split, "expert": f"expert{expert_num}", "expert_num": expert_num, "xml_path": p})

xml_df = pd.DataFrame(xml_paths)
display(xml_df)
missing = xml_df[xml_df["xml_path"].isna()]
if not missing.empty:
    RARE


for _, row in xml_df.iterrows():
    dst = CONFIG_DIR / f"annotations_{row['split']}_{row['expert']}.xml"
    shutil.copy2(row["xml_path"], dst)
    row_path = str(row["xml_path"])
    print(f"Using {row_path} -> {dst}")
xml_df["xml_path"] = xml_df["xml_path"].astype(str)
xml_df.to_csv(REPORTS_DIR / "annotation_xml_files_used.csv", index=False)
xml_df.to_csv(CONFIG_DIR / "annotation_xml_files_used.csv", index=False)

,split,expert,expert_num,xml_path
0,train,expert1,1,/content/project_thermography_equine/data/anno...
1,train,expert2,2,/content/project_thermography_equine/data/anno...
2,valid,expert1,1,/content/project_thermography_equine/data/anno...
3,valid,expert2,2,/content/project_thermography_equine/data/anno...
4,test,expert1,1,/content/project_thermography_equine/data/anno...
5,test,expert2,2,/content/project_thermography_equine/data/anno...


Using /content/project_thermography_equine/data/annotations/annotations_train_expert1.xml -> /content/project_thermography_equine/outputs/config/annotations_train_expert1.xml
Using /content/project_thermography_equine/data/annotations/annotations_train_expert2.xml -> /content/project_thermography_equine/outputs/config/annotations_train_expert2.xml
Using /content/project_thermography_equine/data/annotations/annotations_valid_expert1.xml -> /content/project_thermography_equine/outputs/config/annotations_valid_expert1.xml
Using /content/project_thermography_equine/data/annotations/annotations_valid_expert2.xml -> /content/project_thermography_equine/outputs/config/annotations_valid_expert2.xml
Using /content/project_thermography_equine/data/annotations/annotations_test_expert1.xml -> /content/project_thermography_equine/outputs/config/annotations_test_expert1.xml
Using /content/project_thermography_equine/data/annotations/annotations_test_expert2.xml -> /content/project_thermography_equin

## 4. CVAT parser

In [4]:
BBOX_LABEL = annotation_config.get("hotspot_bbox_label", "hotspot_bbox")
POINT_LABEL = annotation_config.get("hotspot_point_label", "hotspot_point")

def normalize_image_name(name):
    return Path(str(name)).name

def parse_points(points_str):
    pts = []
    for pair in str(points_str).split(';'):
        if not pair.strip():
            continue
        x, y = pair.split(',')[:2]
        pts.append((float(x), float(y)))
    return pts

def parse_cvat_xml(xml_path, split, expert):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    rows = []
    image_rows = []
    for image in root.findall('image'):
        image_name = normalize_image_name(image.attrib.get('name'))
        image_width = float(image.attrib.get('width', np.nan))
        image_height = float(image.attrib.get('height', np.nan))
        image_id_xml = image.attrib.get('id')
        objects = list(image)
        n_bbox = 0
        n_point = 0
        for obj in objects:
            tag = obj.tag
            label = obj.attrib.get('label', '')
            source = obj.attrib.get('source', '')
            if tag == 'box':
                xtl = float(obj.attrib['xtl']); ytl = float(obj.attrib['ytl'])
                xbr = float(obj.attrib['xbr']); ybr = float(obj.attrib['ybr'])
                is_hotspot_bbox = (label == BBOX_LABEL)

                box_label_mismatch = (label == POINT_LABEL)
                if is_hotspot_bbox or box_label_mismatch:
                    n_bbox += 1
                    rows.append({
                        'split': split, 'expert': expert, 'image_name': image_name, 'image_id_xml': image_id_xml,
                        'image_width': image_width, 'image_height': image_height,
                        'object_type': 'bbox', 'label': label, 'source': source,
                        'xtl': xtl, 'ytl': ytl, 'xbr': xbr, 'ybr': ybr,
                        'bbox_width': xbr - xtl, 'bbox_height': ybr - ytl,
                        'bbox_area': max(0, xbr - xtl) * max(0, ybr - ytl),
                        'x': np.nan, 'y': np.nan,
                        'label_mismatch': bool(box_label_mismatch)
                    })
            elif tag == 'points':
                pts = parse_points(obj.attrib.get('points', ''))
                if label == POINT_LABEL and pts:
                    n_point += len(pts)
                    for x, y in pts:
                        rows.append({
                            'split': split, 'expert': expert, 'image_name': image_name, 'image_id_xml': image_id_xml,
                            'image_width': image_width, 'image_height': image_height,
                            'object_type': 'point', 'label': label, 'source': source,
                            'xtl': np.nan, 'ytl': np.nan, 'xbr': np.nan, 'ybr': np.nan,
                            'bbox_width': np.nan, 'bbox_height': np.nan, 'bbox_area': np.nan,
                            'x': x, 'y': y,
                            'label_mismatch': False
                        })
        image_rows.append({
            'split': split, 'expert': expert, 'image_name': image_name,
            'image_width': image_width, 'image_height': image_height,
            'n_bbox': n_bbox, 'n_point': n_point,
            'expert_has_hotspot': (n_bbox > 0 or n_point > 0)
        })
    return pd.DataFrame(rows), pd.DataFrame(image_rows)

## 5. Parse all XML files

In [5]:
object_frames = []
image_frames = []
for _, row in xml_df.iterrows():
    objects_i, images_i = parse_cvat_xml(row['xml_path'], row['split'], row['expert'])
    object_frames.append(objects_i)
    image_frames.append(images_i)

objects_df = pd.concat(object_frames, ignore_index=True) if object_frames else pd.DataFrame()
xml_images_df = pd.concat(image_frames, ignore_index=True) if image_frames else pd.DataFrame()

print("Annotation objects:", len(objects_df))
print("XML image entries:", len(xml_images_df))
display(xml_images_df.groupby(['split','expert','expert_has_hotspot']).size().reset_index(name='n_images'))

if not objects_df.empty and objects_df['label_mismatch'].any():
    print("WARNING: Some box objects had incorrect label names and were parsed defensively. See label_mismatch rows.")
    display(objects_df[objects_df['label_mismatch']].head())

Annotation objects: 384
XML image entries: 260


,split,expert,expert_has_hotspot,n_images
0,test,expert1,False,34
1,test,expert1,True,19
2,test,expert2,False,34
3,test,expert2,True,19
4,train,expert1,True,63
5,train,expert2,True,63
6,valid,expert1,True,14
7,valid,expert2,True,14


## 6. Link XML image entries to master metadata

In [6]:
meta_key_cols = ['image_name', 'split']

xml_linked = xml_images_df.merge(
    master_df[
        [
            'image_name',
            'split',
            'horse_id',
            'relative_image_path',
            'folder_label',
            'label_clinical',
            'label_binary'
        ]
    ],
    on=['image_name', 'split'],
    how='left',
    indicator=True
)

objects_linked = objects_df.merge(
    master_df[
        [
            'image_name',
            'split',
            'horse_id',
            'relative_image_path',
            'folder_label',
            'label_clinical',
            'label_binary'
        ]
    ],
    on=['image_name', 'split'],
    how='left'
)

unmatched_xml_images = xml_linked[xml_linked['_merge'] != 'both'].copy()

print("Unmatched XML image entries:", len(unmatched_xml_images))
if len(unmatched_xml_images):
    display(unmatched_xml_images[['split', 'expert', 'image_name', '_merge']].head(20))

# Images in master metadata with no XML image entry for an expert are allowed.
# Clinical labels come from folders. XML annotations are used only for hotspot localization.
all_expected = []

for expert in ['expert1', 'expert2']:
    tmp = master_df.copy()
    tmp['expert'] = expert
    all_expected.append(
        tmp[
            [
                'split',
                'expert',
                'image_name',
                'horse_id',
                'relative_image_path',
                'folder_label',
                'label_clinical',
                'label_binary'
            ]
        ]
    )

all_expected_df = pd.concat(all_expected, ignore_index=True)

xml_status = all_expected_df.merge(
    xml_images_df[
        [
            'split',
            'expert',
            'image_name',
            'expert_has_hotspot',
            'n_bbox',
            'n_point',
            'image_width',
            'image_height'
        ]
    ],
    on=['split', 'expert', 'image_name'],
    how='left'
)

xml_status['xml_entry_present'] = xml_status['expert_has_hotspot'].notna()
xml_status['expert_has_hotspot'] = xml_status['expert_has_hotspot'].fillna(False).astype(bool)
xml_status['n_bbox'] = xml_status['n_bbox'].fillna(0).astype(int)
xml_status['n_point'] = xml_status['n_point'].fillna(0).astype(int)

# Consensus by image: expert1/expert2 hotspot flags.
consensus = xml_status.pivot_table(
    index=[
        'split',
        'image_name',
        'horse_id',
        'relative_image_path',
        'folder_label',
        'label_clinical',
        'label_binary'
    ],
    columns='expert',
    values='expert_has_hotspot',
    aggfunc='max',
    fill_value=False
).reset_index()

for expert in ['expert1', 'expert2']:
    if expert not in consensus.columns:
        consensus[expert] = False

consensus['has_hotspot_expert1'] = consensus['expert1'].astype(bool)
consensus['has_hotspot_expert2'] = consensus['expert2'].astype(bool)

consensus['has_hotspot_any_expert'] = consensus[
    ['has_hotspot_expert1', 'has_hotspot_expert2']
].any(axis=1)

consensus['has_hotspot_both_experts'] = consensus[
    ['has_hotspot_expert1', 'has_hotspot_expert2']
].all(axis=1)

consensus['expert_hotspot_disagreement'] = (
    consensus['has_hotspot_expert1'] != consensus['has_hotspot_expert2']
)

# Clinically healthy horses may still have expert-marked thermal hotspots.
# Therefore, healthy + hotspot is allowed and flagged, not treated as a clinical-label error.
consensus['healthy_with_expert_hotspot'] = (
    (consensus['label_clinical'] == 'healthy') &
    consensus['has_hotspot_any_expert']
)

# True conflicts are reserved for genuine data errors.
# Under the current protocol, folder-derived clinical labels remain the source of truth.
consensus['annotation_label_conflict'] = False

consensus['annotation_clinical_note'] = np.where(
    consensus['healthy_with_expert_hotspot'],
    'clinically_healthy_with_expert_marked_hotspot',
    'no_clinical_annotation_conflict'
)

print("Consensus rows:", len(consensus))

display(
    consensus.groupby(
        [
            'split',
            'label_clinical',
            'has_hotspot_any_expert',
            'healthy_with_expert_hotspot',
            'annotation_label_conflict'
        ]
    ).size().reset_index(name='n')
)

Unmatched XML image entries: 0
Consensus rows: 347


/tmp/ipykernel_711/600354619.py:84: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  xml_status['expert_has_hotspot'] = xml_status['expert_has_hotspot'].fillna(False).astype(bool)


,split,label_clinical,has_hotspot_any_expert,healthy_with_expert_hotspot,annotation_label_conflict,n
0,test,healthy,False,False,False,34
1,test,healthy,True,True,False,6
2,test,pathological,True,False,False,13
3,train,healthy,False,False,False,179
4,train,pathological,True,False,False,63
5,valid,healthy,False,False,False,38
6,valid,pathological,True,False,False,14


## 7. Expert-specific clean box and point tables

In [7]:
# Keep only objects that match known dataset images.
objects_clean = objects_linked[objects_linked['horse_id'].notna()].copy()
objects_clean['horse_id'] = objects_clean['horse_id'].astype(str)

bboxes = objects_clean[objects_clean['object_type'] == 'bbox'].copy()
points = objects_clean[objects_clean['object_type'] == 'point'].copy()

# BBox validity checks
if not bboxes.empty:
    bboxes['bbox_valid'] = (bboxes['xbr'] > bboxes['xtl']) & (bboxes['ybr'] > bboxes['ytl'])
    bboxes['bbox_within_image'] = (
        (bboxes['xtl'] >= 0) & (bboxes['ytl'] >= 0) &
        (bboxes['xbr'] <= bboxes['image_width']) & (bboxes['ybr'] <= bboxes['image_height'])
    )
else:
    bboxes['bbox_valid'] = []
    bboxes['bbox_within_image'] = []

if not points.empty:
    points['point_within_image'] = (
        (points['x'] >= 0) & (points['y'] >= 0) &
        (points['x'] <= points['image_width']) & (points['y'] <= points['image_height'])
    )
else:
    points['point_within_image'] = []

print('Boxes:', len(bboxes), 'Points:', len(points))
display(bboxes.groupby(['split','expert']).size().reset_index(name='n_boxes'))
display(points.groupby(['split','expert']).size().reset_index(name='n_points'))

Boxes: 192 Points: 192


,split,expert,n_boxes
0,test,expert1,19
1,test,expert2,19
2,train,expert1,63
3,train,expert2,63
4,valid,expert1,14
5,valid,expert2,14


,split,expert,n_points
0,test,expert1,19
1,test,expert2,19
2,train,expert1,63
3,train,expert2,63
4,valid,expert1,14
5,valid,expert2,14


## 8. Save annotation datasets and conflict reports

In [8]:
# Save core outputs for downstream notebooks.
objects_clean.to_csv(CONFIG_DIR / 'hotspot_objects_long.csv', index=False)
objects_clean.to_csv(REPORTS_DIR / 'hotspot_objects_long.csv', index=False)
bboxes.to_csv(CONFIG_DIR / 'hotspot_bboxes.csv', index=False)
bboxes.to_csv(REPORTS_DIR / 'hotspot_bboxes.csv', index=False)
points.to_csv(CONFIG_DIR / 'hotspot_points.csv', index=False)
points.to_csv(REPORTS_DIR / 'hotspot_points.csv', index=False)
xml_status.to_csv(CONFIG_DIR / 'expert_image_hotspot_status.csv', index=False)
xml_status.to_csv(REPORTS_DIR / 'expert_image_hotspot_status.csv', index=False)
consensus.to_csv(CONFIG_DIR / 'hotspot_consensus_by_image.csv', index=False)
consensus.to_csv(REPORTS_DIR / 'hotspot_consensus_by_image.csv', index=False)

unmatched_xml_images.to_csv(REPORTS_DIR / 'unmatched_xml_images.csv', index=False)
label_conflicts = consensus[consensus['annotation_label_conflict']].copy()
label_conflicts.to_csv(REPORTS_DIR / 'annotation_label_conflicts.csv', index=False)
label_conflicts.to_csv(CONFIG_DIR / 'annotation_label_conflicts.csv', index=False)

# Master metadata with annotation flags, without overwriting clinical label source.
master_with_ann = master_df.merge(
    consensus[
        [
            'split',
            'image_name',
            'has_hotspot_expert1',
            'has_hotspot_expert2',
            'has_hotspot_any_expert',
            'has_hotspot_both_experts',
            'expert_hotspot_disagreement',
            'healthy_with_expert_hotspot',
            'annotation_label_conflict',
            'annotation_clinical_note'
        ]
    ],
    on=['split', 'image_name'],
    how='left'
)

bool_cols = [
    'has_hotspot_expert1',
    'has_hotspot_expert2',
    'has_hotspot_any_expert',
    'has_hotspot_both_experts',
    'expert_hotspot_disagreement',
    'healthy_with_expert_hotspot',
    'annotation_label_conflict'
]

for col in bool_cols:
    master_with_ann[col] = master_with_ann[col].fillna(False).astype(bool)

master_with_ann['annotation_clinical_note'] = master_with_ann['annotation_clinical_note'].fillna(
    'no_clinical_annotation_conflict'
)

# Explicit downstream export expected by 02b and later notebooks.
master_with_ann.to_csv(CONFIG_DIR / 'master_metadata_with_annotations.csv', index=False)
master_with_ann.to_csv(REPORTS_DIR / 'master_metadata_with_annotations.csv', index=False)

# Also keep a short machine-readable export manifest for reproducibility checks.
export_manifest = pd.DataFrame([
    {'file': 'master_metadata_with_annotations.csv', 'location': str(CONFIG_DIR / 'master_metadata_with_annotations.csv'), 'n_rows': len(master_with_ann), 'purpose': 'Master metadata joined with expert hotspot flags; required by 02b.'},
    {'file': 'hotspot_objects_long.csv', 'location': str(CONFIG_DIR / 'hotspot_objects_long.csv'), 'n_rows': len(objects_clean), 'purpose': 'Long-format expert annotation objects.'},
    {'file': 'hotspot_bboxes.csv', 'location': str(CONFIG_DIR / 'hotspot_bboxes.csv'), 'n_rows': len(bboxes), 'purpose': 'Expert hotspot bounding boxes.'},
    {'file': 'hotspot_points.csv', 'location': str(CONFIG_DIR / 'hotspot_points.csv'), 'n_rows': len(points), 'purpose': 'Expert hotspot point annotations.'},
    {'file': 'expert_image_hotspot_status.csv', 'location': str(CONFIG_DIR / 'expert_image_hotspot_status.csv'), 'n_rows': len(xml_status), 'purpose': 'Per-image, per-expert hotspot status.'},
    {'file': 'hotspot_consensus_by_image.csv', 'location': str(CONFIG_DIR / 'hotspot_consensus_by_image.csv'), 'n_rows': len(consensus), 'purpose': 'Per-image expert consensus flags.'},
])
export_manifest.to_csv(CONFIG_DIR / 'annotation_import_export_manifest.csv', index=False)
export_manifest.to_csv(REPORTS_DIR / 'annotation_import_export_manifest.csv', index=False)

print('Saved master_metadata_with_annotations.csv to:')
print(' -', CONFIG_DIR / 'master_metadata_with_annotations.csv')
print(' -', REPORTS_DIR / 'master_metadata_with_annotations.csv')
display(master_with_ann.head())


Saved master_metadata_with_annotations.csv to:
 - /content/project_thermography_equine/outputs/config/master_metadata_with_annotations.csv
 - /content/project_thermography_equine/outputs/reports/master_metadata_with_annotations.csv


,horse_id,image_id,image_name,image_stem,image_ext,split,folder_label,label_clinical,label_binary,image_path,...,file_size_bytes,file_sha256,has_hotspot_expert1,has_hotspot_expert2,has_hotspot_any_expert,has_hotspot_both_experts,expert_hotspot_disagreement,healthy_with_expert_hotspot,annotation_label_conflict,annotation_clinical_note
0,0A0F5,0A0F5,0A0F5.jpg,0A0F5,.jpg,test,healthy,healthy,0,/content/project_thermography_equine/data/data...,...,24625,9fd42137a413e2f3a014fa683a06681ba25e6747b32928...,False,False,False,False,False,False,False,no_clinical_annotation_conflict
1,1CGFM,1CGFM,1CGFM.jpg,1CGFM,.jpg,test,healthy,healthy,0,/content/project_thermography_equine/data/data...,...,45753,5257dec22d0001ca8ce4d2c35453e94c9197127df1455e...,False,False,False,False,False,False,False,no_clinical_annotation_conflict
2,1RDFC,1RDFC,1RDFC.jpg,1RDFC,.jpg,test,healthy,healthy,0,/content/project_thermography_equine/data/data...,...,21675,52c7e8f6c67b003f92bc2375f2f36d7cfdfc186576e9ff...,False,False,False,False,False,False,False,no_clinical_annotation_conflict
3,31W0A,31W0A,31W0A.jpg,31W0A,.jpg,test,healthy,healthy,0,/content/project_thermography_equine/data/data...,...,41884,9097ba510c8779f7bc3ca4f1f5cee12e033708b5e5c39a...,False,False,False,False,False,False,False,no_clinical_annotation_conflict
4,6BGWZ,6BGWZ,6BGWZ.jpg,6BGWZ,.jpg,test,healthy,healthy,0,/content/project_thermography_equine/data/data...,...,23589,7a9bfcb4f4af15e4a92ec526f2d38ecffdb50de1b271cd...,False,False,False,False,False,False,False,no_clinical_annotation_conflict


## 9. Readiness decision for downstream modeling

In [9]:
readiness = []

n_healthy_with_expert_hotspot = int(consensus['healthy_with_expert_hotspot'].sum())

readiness.append({
    'criterion': 'metadata_available',
    'status': True,
    'details': f'{len(master_df)} metadata rows'
})

readiness.append({
    'criterion': 'xml_files_loaded',
    'status': len(xml_df) == 6,
    'details': f'{len(xml_df)} XML files'
})

readiness.append({
    'criterion': 'no_unmatched_xml_images',
    'status': len(unmatched_xml_images) == 0,
    'details': f'{len(unmatched_xml_images)} unmatched XML entries'
})

readiness.append({
    'criterion': 'no_true_annotation_label_conflicts',
    'status': len(label_conflicts) == 0,
    'details': (
        f'{len(label_conflicts)} true annotation-label conflicts; '
        f'{n_healthy_with_expert_hotspot} healthy images with expert-marked hotspot '
        f'flagged as allowed by protocol'
    )
})

if not bboxes.empty:
    readiness.append({
        'criterion': 'all_bboxes_valid',
        'status': bool(bboxes['bbox_valid'].all()),
        'details': str(
            bboxes.loc[
                ~bboxes['bbox_valid'],
                ['split', 'expert', 'image_name']
            ].head().to_dict('records')
        )
    })

    readiness.append({
        'criterion': 'all_bboxes_within_image',
        'status': bool(bboxes['bbox_within_image'].all()),
        'details': str(
            bboxes.loc[
                ~bboxes['bbox_within_image'],
                ['split', 'expert', 'image_name']
            ].head().to_dict('records')
        )
    })

if not points.empty:
    readiness.append({
        'criterion': 'all_points_within_image',
        'status': bool(points['point_within_image'].all()),
        'details': str(
            points.loc[
                ~points['point_within_image'],
                ['split', 'expert', 'image_name']
            ].head().to_dict('records')
        )
    })

readiness_df = pd.DataFrame(readiness)

readiness_df.to_csv(
    REPORTS_DIR / 'annotation_import_readiness_report.csv',
    index=False
)

readiness_df.to_csv(
    CONFIG_DIR / 'annotation_import_readiness_report.csv',
    index=False
)

display(readiness_df)

if not readiness_df['status'].all():
    print(
        'WARNING: Some readiness criteria failed. '
        'Classification can use folder labels, but localization/conflict reports need review before final publication.'
    )
else:
    print(
        'Annotation import is ready for downstream analysis. '
        f'{n_healthy_with_expert_hotspot} clinically healthy images contain expert-marked hotspots '
        'and were flagged as allowed by protocol.'
    )

,criterion,status,details
0,metadata_available,True,347 metadata rows
1,xml_files_loaded,True,6 XML files
2,no_unmatched_xml_images,True,0 unmatched XML entries
3,no_true_annotation_label_conflicts,True,0 true annotation-label conflicts; 6 healthy i...
4,all_bboxes_valid,True,[]
5,all_bboxes_within_image,True,[]
6,all_points_within_image,True,[]


Annotation import is ready for downstream analysis. 6 clinically healthy images contain expert-marked hotspots and were flagged as allowed by protocol.
